# Les signaux financiers peuvent-ils prévenir la défaillance d'un fournisseur ?

**Projet 6 – ESME Sudria · agent finance**

Ce notebook présente les résultats en six étapes, autour d'un cas fil rouge : **Nexperia et l'arrêt de production chez Honda** (octobre 2025).

Les calculs sont faits dans `finance_signaux.ipynb`, qui produit les fichiers du dossier `sorties/` ; ce notebook se contente de les lire.

In [ ]:
# Préparation : chargement des résultats et mise en forme des graphiques (rien à présenter ici)
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
SORTIES = Path("sorties")
fichiers = {
    "validation": "finance_validation_cas.csv", "seuils": "finance_seuils.csv", "episodes": "finance_episodes_par_an.csv",
    "brut": "finance_brut.csv", "transmission": "finance_transmission_brent_gazole.csv",
    "fournisseurs": "finance_fournisseurs.csv", "regles": "finance_regles.csv",
}
manquants = [f for f in fichiers.values() if not (SORTIES / f).exists()]
if manquants:
    raise FileNotFoundError("Résultats absents (produits par finance_signaux.ipynb) : " + ", ".join(manquants))

validation = pd.read_csv(SORTIES / fichiers["validation"], parse_dates=["impact", "debut_episode"])
seuils = pd.read_csv(SORTIES / fichiers["seuils"])
episodes = pd.read_csv(SORTIES / fichiers["episodes"], index_col=0)
brut = pd.read_csv(SORTIES / fichiers["brut"], parse_dates=["date"])
transmission = pd.read_csv(SORTIES / fichiers["transmission"], index_col=0)
fournisseurs = pd.read_csv(SORTIES / fichiers["fournisseurs"])
regles = pd.read_csv(SORTIES / fichiers["regles"], parse_dates=["publication", "entree_en_vigueur"])

SEUIL = dict(zip(seuils["signal"], seuils["seuil"]))
NOMS = dict(zip(seuils["signal"], seuils["nom"]))
ARRET_HONDA = pd.Timestamp("2025-10-27")

BLEU, ORANGE, GRIS_CLAIR = "#2a78d6", "#eb6834", "#c9c8c2"
ROUGE, ENCRE, ENCRE_2, GRILLE = "#d03b3b", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRILLE, "axes.labelcolor": ENCRE_2, "xtick.color": ENCRE_2, "ytick.color": ENCRE_2,
    "axes.grid": True, "grid.color": GRILLE, "grid.linewidth": 0.8, "font.size": 10,
    "axes.titlesize": 12.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
})


def dire(texte):
    """Affiche une phrase calculée à partir des résultats."""
    display(Markdown(texte))


def jours_txt(n):
    return f"{int(n)} jour" + ("s" if abs(int(n)) > 1 else "")


def variation(signal, horizon):
    v = brut[brut["signal"] == signal].set_index("date")["valeur"].sort_index()
    if "segment" in brut and brut.loc[brut["signal"] == signal, "segment"].notna().any():
        seg = brut[brut["signal"] == signal].set_index("date")["segment"].sort_index()
        return 100 * v.groupby(seg.to_numpy()).pct_change(horizon, fill_method=None)
    return 100 * v.pct_change(horizon, fill_method=None)

## 1. Le problème

Quand un fournisseur lâche son client, l'usine du client s'arrête. Le 27 octobre 2025, **Honda** arrête une partie de sa production : les puces de **Nexperia**, un fournisseur discret mais présent dans presque toutes les voitures, n'arrivent plus.

L'origine remonte à décembre 2024 : la maison mère de Nexperia, le chinois **Wingtech**, est inscrite sur l'Entity List américaine.

**Notre question** : les signaux financiers et réglementaires auraient-ils permis de le voir venir, et combien de jours avant ?

## 2. Chaque signal a son propre seuil

Une baisse de 30 % en un mois est fréquente pour une petite valeur technologique et exceptionnelle pour un grand équipementier ; une hausse de 20 % est banale pour le gaz et énorme pour l'euro.

Pour chaque signal, on fixe donc un **seuil d'alerte : la variation atteinte seulement dans les 1 % des cas les plus extrêmes** de son histoire (2010-2020).

In [ ]:
quotidiens = seuils[seuils["horizon"] == 20].copy()
quotidiens["ampleur"] = quotidiens["seuil"].abs()
quotidiens = quotidiens.sort_values("ampleur")
couleurs = [ORANGE if s.startswith("bourse_") else BLEU for s in quotidiens["signal"]]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(quotidiens) + 1.2))
ax.barh(quotidiens["nom"], quotidiens["ampleur"], color=couleurs, height=0.6, zorder=2)
for y, (v, s) in enumerate(zip(quotidiens["ampleur"], quotidiens["seuil"])):
    ax.text(v + quotidiens["ampleur"].max() * 0.01, y, f"{s:+.0f} %", va="center", fontsize=9, color=ENCRE_2)
ax.grid(axis="y", visible=False)
ax.set_xlabel("variation sur 20 jours de bourse qui déclenche l'alerte (en valeur absolue, %)")
ax.set_title("Seuil d'alerte de chaque signal (orange : bourse d'un fournisseur)")
plt.tight_layout()
plt.show()

bas, haut = quotidiens.iloc[0], quotidiens.iloc[-1]
dire(f"Le seuil va de **{bas['seuil']:+.0f} %** ({bas['nom']}) à **{haut['seuil']:+.0f} %** ({haut['nom']}) : "
     f"un seuil unique n'aurait aucun sens.")

## 3. Nexperia : ce que montraient la bourse et les règles

Deux signaux pouvaient alerter : la **bourse de Wingtech**, et la **règle américaine** publiée au Federal Register qui inscrit Wingtech sur l'Entity List.

In [ ]:
regle = validation[validation["signal"] == "federal_register"]
sanction = validation[validation["signal"] == "sanction_maison_mere"]
date_regle = regle["debut_episode"].iloc[0] if len(regle) and pd.notna(regle["debut_episode"].iloc[0]) else None
bourse = validation[validation["signal"] == "bourse_wingtech"]
bourse_ok = bool(len(bourse)) and bourse["alerte_avant_impact"].iloc[0] == True

fig, ax = plt.subplots(figsize=(10, 4))
if "bourse_wingtech" in SEUIL:
    v = variation("bourse_wingtech", 20)
    v = v[(v.index >= "2024-06-01") & (v.index <= "2025-12-31")]
    ax.plot(v.index, v, color=BLEU, lw=1.3, label="bourse de Wingtech, variation sur 20 jours (%)")
    ax.axhline(SEUIL["bourse_wingtech"], color=BLEU, ls="--", lw=1, label="seuil d'alerte de la bourse")
if date_regle is not None:
    ax.axvline(date_regle, color=ORANGE, lw=2, label=f"règle US publiée ({date_regle:%d/%m/%Y})")
ax.axvline(ARRET_HONDA, color=ROUGE, lw=2, label="arrêt chez Honda (27/10/2025)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax.set_title("Nexperia : la règle et la bourse alertent" if bourse_ok else "Nexperia : la règle alerte, la bourse non")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

if date_regle is not None:
    avance_regle = jours_txt((ARRET_HONDA - date_regle).days)
    if bourse_ok:
        dire(f"La règle américaine est publiée **{avance_regle} avant** l'arrêt chez Honda, et la bourse de Wingtech "
             f"alerte aussi ({jours_txt(bourse['jours_avance'].iloc[0])} avant) : les deux signaux convergent.")
    else:
        dire(f"La règle américaine est publiée **{avance_regle} avant** l'arrêt chez Honda ; la bourse de Wingtech "
             f"**n'a pas franchi son seuil**. Pour ce fournisseur, ce sont **les règles et les sanctions** qui alertent, pas le marché.")

## 4. Nyrstar : quand l'énergie coupe un fournisseur

En août 2022, Nyrstar annonce l'arrêt de sa fonderie de zinc de Budel (Pays-Bas) à partir du 1er septembre : l'énergie est devenue trop chère pour produire.

In [ ]:
ny = validation[validation["signal"] == "gaz_europe"]
if len(ny) and "gaz_europe" in SEUIL:
    impact = ny["impact"].iloc[0]
    v = variation("gaz_europe", 1)
    v = v[(v.index >= "2021-06-01") & (v.index <= "2022-12-31")]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(v.index, v, width=20, color=[ORANGE if x >= SEUIL["gaz_europe"] else GRIS_CLAIR for x in v], zorder=2)
    ax.axhline(SEUIL["gaz_europe"], color=ENCRE_2, ls="--", lw=1, label="seuil d'alerte")
    ax.axvline(impact, color=ROUGE, lw=2, label="arrêt de la fonderie (01/09/2022)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
    ax.set_ylabel("variation sur un mois (%)")
    ax.set_title("Prix du gaz en Europe : variation d'un mois sur l'autre")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    r = ny.iloc[0]
    if r["alerte_avant_impact"]:
        dire(f"Le gaz passe en alerte **{jours_txt(r['jours_avance'])} avant** l'arrêt (hausse de {r['valeur_extreme']:.0f} % en un mois). "
             f"Le prix d'un mois est daté de son dernier jour et publié quelques jours plus tard : l'avance réelle est de quelques jours de moins.")
    else:
        dire("Le gaz n'a pas franchi son seuil avant l'arrêt.")

## 5. La même méthode sur quatre crises de fournisseurs

| Fournisseur | Client touché | Ce qui s'est passé |
| --- | --- | --- |
| Nyrstar Budel (zinc) | Acheteurs européens de zinc | Fonderie arrêtée le 01/09/2022 à cause du coût de l'énergie |
| Fournisseurs chinois de composants | Tesla Grünheide | Usine arrêtée du 29/01 au 11/02/2024, composants bloqués par la crise de la mer Rouge |
| Wolfspeed (carbure de silicium) | Renesas | Faillite le 30/06/2025 |
| Nexperia (puces) | Honda | Arrêt de production le 27/10/2025 |

In [ ]:
LIBELLES = {"gaz_europe": "gaz Europe", "gscpi": "pression supply chain (GSCPI)", "drewry_wci": "fret Drewry",
            "bourse_wolfspeed": "bourse du fournisseur", "bourse_wingtech": "bourse de la maison mère",
            "sanction_maison_mere": "sanction sur la maison mère", "federal_register": "règle US publiée"}
v = validation.copy()
v["libelle"] = v["cas"].str.split(" \\(").str[0] + " · " + v["signal"].map(LIBELLES).fillna(v["signal"])
v["avance"] = np.where(v["alerte_avant_impact"] == True, v["jours_avance"], 0)
v = v.sort_values(["cas", "avance"], ascending=[False, True]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 0.45 * len(v) + 1.2))
ax.barh(v["libelle"], v["avance"], color=[BLEU if a > 0 else GRIS_CLAIR for a in v["avance"]], height=0.6, zorder=2)
for y, (a, ok) in enumerate(zip(v["avance"], v["alerte_avant_impact"])):
    ax.text(a + v["avance"].max() * 0.01, y, jours_txt(a) if ok == True else "pas d'alerte avant", va="center", fontsize=9, color=ENCRE_2)
ax.grid(axis="y", visible=False)
ax.set_xlabel("jours d'avance sur l'impact chez le client")
ax.set_title("Avance obtenue sur chaque crise, signal par signal")
plt.tight_layout()
plt.show()

par_cas = validation.groupby("cas")["alerte_avant_impact"].apply(lambda s: (s == True).any())
dire(f"**{par_cas.sum()} crises sur {len(par_cas)}** ont été vues venir par au moins un signal ; "
     f"**{(validation['alerte_avant_impact'] == True).sum()} tests sur {len(validation)}** donnent une alerte avant l'impact.")

## 6. Combien d'alertes par an ?

Un signal qui alerte toutes les semaines n'est pas utilisable. On compte combien de fois chaque signal passe en alerte, en moyenne, chaque année.

In [ ]:
ep = episodes["episodes_par_an"].sort_values()
fig, ax = plt.subplots(figsize=(9, 0.38 * len(ep) + 1.2))
ax.barh([NOMS.get(s, s) for s in ep.index], ep.values, color=BLEU, height=0.6, zorder=2)
for y, val in enumerate(ep.values):
    ax.text(val + ep.max() * 0.01, y, f"{val:.1f}", va="center", fontsize=9, color=ENCRE_2)
ax.grid(axis="y", visible=False)
ax.set_xlabel("épisodes d'alerte par an")
ax.set_title("Le bruit de chaque signal")
plt.tight_layout()
plt.show()
total = ep.sum()
dire(f"Au total, **{total:.0f} épisodes d'alerte par an** pour {len(ep)} signaux, soit environ {total / 12:.1f} par mois.")

## 7. Les fournisseurs aujourd'hui

L'état de chaque fournisseur suivi : bourse, maisons mères, sanctions, procédures collectives.

In [ ]:
colonnes = ["fournisseur", "pays", "bourse_20j_%", "maisons_meres", "sanctions", "procedures_collectives", "en_alerte", "alertes"]
display(fournisseurs[[c for c in colonnes if c in fournisseurs.columns]].fillna(""))
n = int(fournisseurs["en_alerte"].sum())
dire(f"**{n} fournisseur(s) sur {len(fournisseurs)}** en alerte aujourd'hui.")

## Ce qu'on retient

In [ ]:
tr = transmission.loc["EU"] if "EU" in transmission.index else (transmission.iloc[0] if len(transmission) else None)
point_tr = (f"**Le gazole suit le Brent avec {int(tr['decalage_semaines'])} semaine(s) de retard** : c'est le temps dont dispose un acheteur "
            f"pour renégocier ses transports." if tr is not None else "Le délai Brent → gazole n'a pas pu être mesuré.")
ok = lambda sig: bool((validation.loc[validation["signal"] == sig, "alerte_avant_impact"] == True).any())
if ok("drewry_wci") and not ok("gscpi"):
    point_fret = "**Un indice mondial ne suffit pas pour une crise régionale** : le fret Asie–Europe a vu la crise de la mer Rouge, le GSCPI non."
elif ok("gscpi"):
    point_fret = "**Le fret a vu venir la crise de la mer Rouge**, y compris l'indice mondial GSCPI."
else:
    point_fret = "**Le fret n'a pas permis d'anticiper la crise de la mer Rouge** avec les données disponibles."
rates = ", ".join(par_cas[~par_cas].index)
point_cas = (f"**{par_cas.sum()} crises de fournisseurs sur {len(par_cas)} vues venir.**"
             + (f" Non vue(s) : {rates}." if rates else ""))
dire(f"""
1. {point_cas}
2. **Chaque signal a son rôle.** Les règles et les sanctions alertent le plus tôt (plusieurs mois) ; la bourse d'un fournisseur prévient d'une faillite ; l'énergie et le fret préviennent d'une hausse de coût ou d'un retard.
3. {point_fret}
4. {point_tr}
5. **Les signaux de marché sont un risque de fond** (environ {total:.0f} alertes par an au total) ; une sanction, une règle ou une procédure collective qui vise un fournisseur précis est une alerte en soi.
""")

**Limites**

- Quatre crises montrent que la méthode **peut** marcher, pas qu'elle est fiable en toutes circonstances.
- Les prix mensuels (gaz, zinc, nickel, prix de production) sont publiés quelques jours après la fin du mois : l'avance réelle est un peu plus courte que l'avance affichée.
- L'historique boursier d'une entreprise disparaît parfois après une faillite (Wolfspeed) : on s'appuie alors sur des valeurs publiées.
- Les règles UE sont suivies à la main ; seules les règles américaines sont collectées automatiquement.

**Prochaine étape** : brancher la fiche de chaque fournisseur dans le graphe commun, pour que le moteur de risque combine signaux de marché, sanctions et dépendance du client.

---

*Données : yfinance (Yahoo), Banque mondiale (Pink Sheet, CC BY 4.0), Commission européenne (Weekly Oil Bulletin, sanctions financières), BCE, Eurostat, Réserve fédérale de New York (GSCPI), Federal Register (domaine public), trade.gov (Consolidated Screening List), GLEIF, BODACC. Version sans code :* `jupyter nbconvert --to html --no-input presentation_finance.ipynb`